[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/your-notebook.ipynb)

In [39]:
!pip install -q transformers torch gradio accelerate datasets

In [40]:
import torch
import warnings
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    AutoTokenizer, AutoModelForSequenceClassification,
    pipeline
)
import gradio as gr
import numpy as np
from datetime import datetime

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("Setting up models...")

Setting up models...


In [41]:
class NewsAI:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load generation model (GPT-2 Medium for better quality)
        print("Loading text generation model...")
        self.gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
        self.gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2-medium").to(self.device)

        # Set padding token
        if self.gpt2_tokenizer.pad_token is None:
            self.gpt2_tokenizer.pad_token = self.gpt2_tokenizer.eos_token

        # Load detection model (using a pre-trained fake news detector)
        print("🔍 Loading fake news detection model...")
        try:
            # Try to use a specialized fake news detection model
            self.detector = pipeline(
                "text-classification",
                model="hamzab/roberta-fake-news-classification",
                device=0 if torch.cuda.is_available() else -1
            )
        except:
            # Fallback to BERT-based classification
            print("Specialized model not available, using BERT fallback...")
            self.bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
            self.bert_model = AutoModelForSequenceClassification.from_pretrained(
                "bert-base-uncased", num_labels=2
            ).to(self.device)
            self.detector = None

        print("Models loaded successfully!")

    def generate_fake_news(self, prompt, max_length=300, temperature=0.8):
        """Generate fake news article from prompt"""
        try:
            if not prompt or len(prompt.strip()) < 5:
                return "Please provide a meaningful prompt (at least 5 characters)"

            # Add context to make it more news-like
            enhanced_prompt = f"Breaking News: {prompt.strip()}"

            inputs = self.gpt2_tokenizer.encode(
                enhanced_prompt,
                return_tensors="pt",
                max_length=100,
                truncation=True
            ).to(self.device)

            with torch.no_grad():
                outputs = self.gpt2_model.generate(
                    inputs,
                    max_length=max_length,
                    num_return_sequences=1,
                    no_repeat_ngram_size=3,
                    do_sample=True,
                    temperature=temperature,
                    top_k=40,
                    top_p=0.9,
                    pad_token_id=self.gpt2_tokenizer.eos_token_id,
                    early_stopping=True
                )

            generated_text = self.gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Clean up the text
            generated_text = generated_text.replace(enhanced_prompt, "").strip()
            if not generated_text:
                generated_text = self.gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)

            return f"📰 Generated Article:\n\n{enhanced_prompt} {generated_text}"

        except Exception as e:
            return f" Error generating text: {str(e)}"

    def detect_fake_news(self, text):
        """Detect if news is fake or real"""
        try:
            if not text or len(text.strip()) < 10:
                return " Please provide meaningful text (at least 10 characters)"

            if self.detector:
                # Use specialized fake news detector
                result = self.detector(text[:512])  # Limit text length
                label = result[0]['label'].upper()
                confidence = result[0]['score']

                if 'FAKE' in label or 'FALSE' in label:
                    emoji = "🚨"
                    status = "FAKE NEWS"
                else:
                    emoji = "✅"
                    status = "LIKELY REAL"

                return f"{status}\nConfidence: {confidence:.1%}\n\n"

            else:
                # Fallback BERT classification
                inputs = self.bert_tokenizer(
                    text,
                    return_tensors="pt",
                    truncation=True,
                    padding=True,
                    max_length=512
                ).to(self.device)

                with torch.no_grad():
                    outputs = self.bert_model(**inputs)

                logits = outputs.logits
                probabilities = torch.softmax(logits, dim=1)
                predicted_class = torch.argmax(logits, dim=1).item()
                confidence = probabilities[0][predicted_class].item()

                if predicted_class == 0:
                    return f"🚨 FAKE NEWS\nConfidence: {confidence:.1%}\n\n"
                else:
                    return f"✅ LIKELY REAL\nConfidence: {confidence:.1%}\n\n"

        except Exception as e:
            return f"❌ Error detecting news: {str(e)}"

In [42]:
news_ai = NewsAI()

Using device: cpu
Loading text generation model...
🔍 Loading fake news detection model...


Device set to use cpu


Models loaded successfully!


In [43]:
def createinterface(customcss=None):
    customcss = customcss or """
    .gradio-container {
        max-width: 900px !important;
        margin: auto;
        background: linear-gradient(135deg, #56CC9D 0%, #FAD7A1 100%);
        padding: 30px;
        border-radius: 20px;
    }
    .tab-nav { border-radius: 10px; }
    .selected { background: rgba(86,204,157,0.2); }
    .gr-button {
        background: linear-gradient(45deg, #FF6B6B, #FFD93D) !important;
        color: black !important;
        font-weight: bold !important;
        border-radius: 18px !important;
        padding: 10px 25px !important;
    }
    .gr-button:hover { transform: scale(1.07) !important; }
    """
    with gr.Blocks(css=customcss, title="Custom News AI") as interface:
        gr.Markdown("# Custom AI News Generator & Detector")
        gr.Markdown("Educational tool: Generated articles are fictional. For demonstration only.")

        with gr.Tab("Generate Fake News", elem_classes="gr-panel"):
            gr.Markdown("Enter a news prompt and tweak parameters.")
            with gr.Row():
                with gr.Column():
                    geninput = gr.Textbox(label="News Prompt", placeholder="Eg: Scientists discover flying elephants")
                    genlength = gr.Slider(100, 500, value=250, label="Length")
                    gentemp = gr.Slider(0.5, 1.2, value=1.0, step=0.05, label="Creativity")
                    genbutton = gr.Button("Generate", variant="primary")
                with gr.Column():
                    genoutput = gr.Textbox(label="Generated Content", lines=12, max_lines=18)
            genbutton.click(lambda prompt, length, temp: news_ai. generate_fake_news(prompt, int(length), temp),
                            inputs=[geninput, genlength, gentemp], outputs=genoutput)

        with gr.Tab("Detect Fake News", elem_classes="gr-panel"):
            gr.Markdown("Paste text for AI analysis.")
            with gr.Row():
                detectinput = gr.Textbox(label="Paste Text", lines=8, max_lines=12)
                detectbutton = gr.Button("Analyze", variant="primary")
                detectoutput = gr.Textbox(label="Result", lines=5)
            detectbutton.click(lambda text:news_ai.detect_fake_news(text),
                               inputs=detectinput, outputs=detectoutput)

        with gr.Tab("About", elem_classes="gr-panel"):
            gr.Markdown("## About This Tool\n- Purpose: Demo AI news generation/detection.\n- Models: GPT-2, RoBERTa fine-tuned.\n- Note: Fictional output, not for real reporting.")

    return interface




In [44]:
if __name__ == "__main__":
    demo = createinterface()
    demo.launch(
        share=True,
        inbrowser=True,
        show_error=True,
        debug=False
    )
    print("🎉 Interface launched! Click the public URL above to access the app.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57f021853170eeb8d6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎉 Interface launched! Click the public URL above to access the app.
